# Customer segmentation
### Jimmy Azar

## Introduction

We'll explore customer segmentation by statistical methods. In particular, we'll use k-means clustering and select the optimal number of clusters by the silhouette score. We'll then identify the most-valued segment and its properties.

## Explore the data

Our data consists of real purchases made between 01.12.2010 and 09.12.2011 at a retail company that sells online gift items.

In [1]:
import pandas as pd
import numpy as np

path_to_file = '../data/invoices.csv'
data = pd.read_csv(path_to_file)

print(data.head())
data.shape

  invoice_number stock_code                          description  quantity  \
0         536365     85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365      71053                  WHITE METAL LANTERN         6   
2         536365     84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365     84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365     84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          invoice_date  unit_price  customer_id         country  
0  2010-12-01 08:26:00        2.55      17850.0  United Kingdom  
1  2010-12-01 08:26:00        3.39      17850.0  United Kingdom  
2  2010-12-01 08:26:00        2.75      17850.0  United Kingdom  
3  2010-12-01 08:26:00        3.39      17850.0  United Kingdom  
4  2010-12-01 08:26:00        3.39      17850.0  United Kingdom  


(541909, 8)

## Data preparation

By inspecting the "quantity" field, we notice that there are any quantities that aren't positive. Negative values signify that the order was canceled, and these transactions can be ignored for our purpose. 

Next, we inspect the "customer_id" field. Notably, there are missing values. So, we remove any observation containing NaN.

In [2]:
# data preparation
print(data.loc[data['quantity']<0].head(6))
df = data.loc[data['quantity']>0]
df.loc[df['customer_id'].isna()]
df = df.dropna(subset=['customer_id']) 
df.shape

    invoice_number stock_code                       description  quantity  \
141        C536379          D                          Discount        -1   
154        C536383     35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235        C536391      22556     PLASTERS IN TIN CIRCUS PARADE       -12   
236        C536391      21984   PACK OF 12 PINK PAISLEY TISSUES       -24   
237        C536391      21983   PACK OF 12 BLUE PAISLEY TISSUES       -24   
238        C536391      21980  PACK OF 12 RED RETROSPOT TISSUES       -24   

            invoice_date  unit_price  customer_id         country  
141  2010-12-01 09:41:00       27.50      14527.0  United Kingdom  
154  2010-12-01 09:49:00        4.65      15311.0  United Kingdom  
235  2010-12-01 10:24:00        1.65      17548.0  United Kingdom  
236  2010-12-01 10:24:00        0.29      17548.0  United Kingdom  
237  2010-12-01 10:24:00        0.29      17548.0  United Kingdom  
238  2010-12-01 10:24:00        0.29      17548.0  U

(397924, 8)

We use datetime conversion to store dates as numeric values.

The earliest date is 2010-12-01 and the most recent is 2011-12-09. The data therefore spans around a year. Let's remove data from the last (incomplete) month of December 2011.

In [3]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['invoice_date'].min()
df['invoice_date'].max()

df = df.loc[df['invoice_date']<'2011-12-01']
print(df.shape)

df['quantity'].value_counts();

(380620, 8)


## Feature extraction 

We are interested in the revenue made for each type of item purchased. Therefore, we add a column named "sales" to the data frame consisting of the *quantity* value multiplied by the *unit price* value for each observation.

For each unique customer ID, we derive three features: total sales, order count, and average sales. Total sales is the sum of all sales for a given customer. Order count is the total number of unique orders made by a customer and is measured using unique invoice dates. Average sales is the total sales for a given customer divided by the order count for that customer.

The scales of the features we've derived differ largely. Moreover, most customers will have relatively low sales and order count and can expect to form one dense cluster, whereas a few outliers with large feature values will form their own small clusters. To mitigate this problem, we will use rank data over the original features. Each feature value is replaced by its corresponding rank within its own column. Ranks proceed in increasing order with the smallest value having a rank of 1.

Although using rank data ensures feature scales are comparable, nonetheless, we will normalize the data to have zero mean and unit variance.

In [4]:
from sklearn.preprocessing import scale

# feature extraction
df['sales'] = df['quantity'] * df['unit_price']

df_customers = df.groupby(['customer_id']).aggregate({'sales': 'sum', 'invoice_date': 'nunique'})
df_customers = df_customers.rename(columns={'sales':'total_sales', 'invoice_date':'order_count'})
df_customers['avg_sales'] = df_customers['total_sales']/df_customers['order_count']

df_customers.describe()
df_customers.shape

df_rank = df_customers.rank(method='first')

df_norm = pd.DataFrame(scale(df_rank), index=df_rank.index, columns=df_rank.columns) 

df_norm.describe().loc[['mean','std']]

,total_sales,order_count,avg_sales
mean,-6.612776e-18,0.000000,0.000000
std,1.000116e+00,1.000116,1.000116


## Customer segmentation by k-means clustering

We are interested in clustering our data into at least 4 clusters. We'll use k-means clustering. We'll also show the count (or size) of each resulting cluster.

In [5]:
from sklearn.cluster import KMeans

model = KMeans(n_clusters=4, max_iter=100, n_init=10).fit(df_norm)
model.labels_
model.cluster_centers_

df_customers['label'] = model.labels_
df_customers.groupby(['label'])['order_count'].count()

label
0    1134
1    1147
2    1060
3     957
Name: order_count, dtype: int64

For each cluster center representing a segment, we can compute the average feature values. We find the profile of each cluster in terms of actual (not normalized) total sales, order count, and average sales of its corresponding center.

In [6]:
df_customers.groupby(['label']).mean()

,total_sales,order_count,avg_sales
label,,,
0,5628.041773,8.492063,615.531887
1,190.773505,1.321709,152.161004
2,764.249378,1.326415,616.778657
3,1026.221726,5.347962,207.339628


### Optimal number of clusters by silhouette

So far we have selected the number of clusters, $k=4$, arbitrarily. Instead, we can optimize the selection of $k$ over a range of values. The criterion we'll use is the silhouette coefficient which lies in $[-1,1]$. We automatically select $k$ which maximizes this coefficient. Since we are interested in having at least 4 clusters, we choose a range for $k$ from 4 to 10 clusters to test out. Once the data has been clustered, a silhouette value, $s(i)$ for a given object $i$ is given by:

$$s(i) := \dfrac{b(i) - a(i)}{max(a(i),b(i))}$$

$a(i)$ is the average dissimilarity between object $i$ and all other points (excluding $i$) which belong to the cluster that $i$ belongs to. That is, $a(i) := \dfrac{1}{|C_i|-1}\sum_{j \in C_i, i\neq j} d(i,j)$. All dissimilarities can be computed by constructing the distance matrix $d(i,j)$. We divide by $|C_i|-1$ because we do not include the distance $d(i,i)$ in the sum. In case object $i$ is the only object in its cluster, then we set $s(i) := 0$. The variable $b(i)$ is the minimum among all average dissimilarities measured between object $i$ and the instances of another cluster which $i$ does not belong to. That is, $b(i) := \underset{C}{\operatorname{min}} d(i,C)$ where $d(i,C)$ is the average dissimilarity of $i$ to all instances in cluster $C$ which $i$ does not belong to. The cluster with the smallest $b(i)$ is the "neighboring cluster" of $i$: it is the next best fit cluster for point $i$.

$s(i)$ close to 1 means that $a(i) << b(i)$, and so $i$ is closer or more "similar" to its own cluster than its neighboring cluster; thus $i$ is well-labeled. If $s(i)$ is close to $-1$, then $a(i) >> b(i)$, and so $i$ is better off belonging to its neighboring cluster and is thus poorly labeled. If $s(i) = 0$, then $i$ is equally likely to fall into either its own cluster or its neighboring cluster, so $i$ lies between two clusters. 

The silhouette value $s(i)$ applies to each clustered observation. The silhouette coefficient to assess the overall clustering is taken to be the mean of all silhouette values $S := \dfrac{1}{n}\sum_i s(i)$. We select $k$ that maximizes this coefficient.

Once we've identified the optimal value for $k$, we re-run k-means using this value. Finally, we identify which cluster (label) represents our most-valued customers, i.e., that with the highest average sales.

In [7]:
# optimal number of clusters by silhouette
from sklearn.metrics import silhouette_score

k_values, scores = range(4,11), []
for k in k_values:
    model = KMeans(n_clusters=k, max_iter = 100, n_init=10).fit(df_norm)
    score = silhouette_score(df_norm, model.labels_)
    scores.append(score)    

k_optimal = k_values[np.argmax(scores)]
print(f'optimal k: {k_optimal}')

model = KMeans(n_clusters=k_optimal, max_iter = 100, n_init=10).fit(df_norm)

df_customers['label'] = model.labels_

high_value_label = df_customers.groupby('label').mean()['avg_sales'].argmax()
print(f'high value label: {high_value_label}')

df_norm['label'] = model.labels_

optimal k: 4
high value label: 0


## Identify high-valued segment

Using the identified cluster label corresponding to high-valued customers, we provide a summary of the (non-normalized) features corresponding to this segment. Also, we list the customer IDs of all high-valued customers belonging to this cluster. Finally, we are interested in displaying the item descriptions and their count which these customers have bought.

We sort the data frame items to display the descriptions from those with highest count at the top, and we show the six most frequent items bought by the high-valued customer segment.

In [8]:
high_valued_customers = df_customers.loc[df_customers['label'] == high_value_label]
high_valued_customers.describe()

vip = np.array(high_valued_customers.index)

df_vip = df.loc[df['customer_id'].isin(vip)]

items = df_vip.groupby('description').aggregate({'description': 'count'})
items = items.rename(columns= {'description': 'count'})

items = items.sort_values(by=["count"], ascending = False)
items.head(6)

,count
description,
JUMBO BAG RED RETROSPOT,1143
REGENCY CAKESTAND 3 TIER,1084
WHITE HANGING HEART T-LIGHT HOLDER,1077
LUNCH BAG RED RETROSPOT,937
PARTY BUNTING,868
ASSORTED COLOUR BIRD ORNAMENT,826
